# Asset Sirius — Dimensionamento das EC2
## Programação linear inteira: capacidade do front-end e do back-end

Este notebook compara tipos reais da família EC2 usados como referência para as duas instâncias definidas no CloudFormation do projeto:

- `asset-sirius-ec2-nodejs`: camada de aplicação, analisada como front-end e back-end;
- `asset-sirius-ec2-grafana`: camada de dashboards e visualização.

> Os preços são aproximações em US$ para a região us-east-1, Linux sob demanda, considerando 30 dias de uso contínuo. As capacidades são estimativas didáticas e devem ser substituídas por resultados de teste de carga.

## 1. Infraestrutura e premissas

O template `script_cloudformation/compute.yaml` cria duas EC2 com o mesmo parâmetro `InstanceType`, cujo padrão é `t2.micro`. Para estudar o dimensionamento, comparamos três tipos reais da mesma família:

| Tipo EC2 | vCPU | Memória | Custo estimado/hora | Custo estimado/mês |
|---|---:|---:|---:|---:|
| t2.micro | 1 | 1 GiB | US$ 0,0116 | US$ 8,35 |
| t2.small | 1 | 2 GiB | US$ 0,0230 | US$ 16,56 |
| t2.medium | 2 | 4 GiB | US$ 0,0464 | US$ 33,41 |

A mesma escolha é avaliada por duas métricas de capacidade:

- **Front-end:** usuários simultâneos atendidos pela EC2 Node.js;
- **Back-end:** requisições por segundo processadas pela EC2 Node.js.

**Premissas do modelo:**

- cada cálculo escolhe exatamente um tipo de EC2;
- orçamento máximo de US$ 0,05 por hora para cada EC2;
- mínimo de 1 GiB de memória para o front-end e 2 GiB para o back-end;
- capacidades estimadas: 100/150/300 usuários simultâneos e 15/30/60 requisições por segundo;
- a referência visual indica US$ 446,95/mês para a infraestrutura base; esse valor é tratado como custo compartilhado, sem atribuí-lo artificialmente a uma EC2;
- VPC, RDS, S3 e demais componentes permanecem fora da decisão de dimensionamento das EC2.

## 2. Dois modelos de programação linear inteira

Para cada tipo de EC2, usamos uma variável binária $x_i$. Ela vale 1 quando o tipo $i$ é escolhido e 0 caso contrário.

Os dois modelos usam as mesmas restrições de escolha, custo e memória:

$$
\sum_i x_i = 1
$$

$$
\sum_i custo_i \cdot x_i \le 0{,}05
$$

$$
\sum_i memoria_i \cdot x_i \ge memoria\ mínima
$$

$$
 x_i \in \{0,1\}
$$

In [2]:
%pip install numpy pandas scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 34.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 42.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 41.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]2m2/3 [pandas]

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds

ec2 = pd.DataFrame({
    "tipo": ["t2.micro", "t2.small", "t2.medium"],
    "vcpu": [1, 1, 2],
    "memoria_gib": [1, 2, 4],
    "custo_hora": [0.0116, 0.0230, 0.0464],
    "custo_mes": [8.35, 16.56, 33.41],
    "usuarios": [100, 150, 300],
    "requisicoes_segundo": [15, 30, 60]
})

orcamento_hora = 0.05


def resolver_modelo(coluna_objetivo, memoria_minima):
    """
    Resolve o modelo binário de escolha de uma instância EC2.

    Variáveis:
        x[i] = 1 se o tipo EC2 i for escolhido; 0 caso contrário.
    """

    objetivo = -ec2[coluna_objetivo].to_numpy(dtype=float)

    matriz_restricoes = np.array([
        np.ones(len(ec2)),              # exatamente uma EC2
        ec2["custo_hora"],              # limite de custo
        ec2["memoria_gib"]              # memória mínima
    ])

    limites_inferiores = np.array([
        1,
        -np.inf,
        memoria_minima
    ])

    limites_superiores = np.array([
        1,
        orcamento_hora,
        np.inf
    ])

    restricoes = LinearConstraint(
        matriz_restricoes,
        limites_inferiores,
        limites_superiores
    )

    resultado = milp(
        c=objetivo,
        integrality=np.ones(len(ec2)),
        bounds=Bounds(0, 1),
        constraints=restricoes
    )

    if not resultado.success:
        raise ValueError(f"Modelo inviável: {resultado.message}")

    indice_escolhido = np.argmax(resultado.x)
    escolhido = ec2.iloc[indice_escolhido].copy()

    return escolhido, resultado

### Modelo do front-end

A função objetivo maximiza os usuários simultâneos:

$$
\text{Max } Z_f = 100x_{micro} + 150x_{small} + 300x_{medium}
$$



In [4]:
front_end, resultado_front = resolver_modelo(
    coluna_objetivo="usuarios",
    memoria_minima=1
)

print("Resultado do modelo do front-end")
print(f"Tipo escolhido: {front_end['tipo']}")
print(f"Usuários simultâneos: {front_end['usuarios']}")
print(f"Memória: {front_end['memoria_gib']} GiB")
print(f"Custo por hora: US$ {front_end['custo_hora']:.4f}")
print(f"Custo mensal: US$ {front_end['custo_mes']:.2f}")

Resultado do modelo do front-end
Tipo escolhido: t2.medium
Usuários simultâneos: 300
Memória: 4 GiB
Custo por hora: US$ 0.0464
Custo mensal: US$ 33.41


### Modelo do back-end

A função objetivo maximiza a capacidade de requisições por segundo:

$$
\text{Max } Z_b = 15x_{micro} + 30x_{small} + 60x_{medium}
$$

In [5]:
back_end, resultado_back = resolver_modelo(
    coluna_objetivo="requisicoes_segundo",
    memoria_minima=2
)

print("Resultado do modelo do back-end")
print(f"Tipo escolhido: {back_end['tipo']}")
print(f"Requisições por segundo: {back_end['requisicoes_segundo']}")
print(f"Memória: {back_end['memoria_gib']} GiB")
print(f"Custo por hora: US$ {back_end['custo_hora']:.4f}")
print(f"Custo mensal: US$ {back_end['custo_mes']:.2f}")

Resultado do modelo do back-end
Tipo escolhido: t2.medium
Requisições por segundo: 60
Memória: 4 GiB
Custo por hora: US$ 0.0464
Custo mensal: US$ 33.41
